In [4]:
import pandas as pd
import networkx as nx

# Load data
df = pd.read_csv('data/enron.csv')
df = df[['From', 'To']].dropna()

print(df.head())  # Check the first few rows to ensure correct loading

# Weighted Directed Graph
edges = df.groupby(['From', 'To']).size().reset_index(name='weight')
Gw = nx.from_pandas_edgelist(
    edges, 
    'From', 
    'To', 
    edge_attr='weight', 
    create_using=nx.DiGraph()
)

####################################################################################################

# Compute Centralities
print("\nComputing centralities...")

# Unweighted degree (total connections)
deg = nx.degree_centrality(Gw)

# Weighted Betweenness (Approximated with k=200)
# Note: NetworkX uses weight as "distance," so for "importance," we use 1/weight
for u, v, d in Gw.edges(data=True):
    d['inv_w'] = 1.0 / d['weight']
bet = nx.betweenness_centrality(Gw, k=200, normalized=True, weight='inv_w', seed=42)

# Weighted Eigenvector
eig = nx.eigenvector_centrality(Gw, max_iter=2000, tol=1e-6, weight="weight")

####################################################################################################

# Extract and Print Top 10
def print_top_10(centrality_dict, name):
    top_nodes = sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\nTop 10 {name}:")
    for node, val in top_nodes:
        print(f"{node}: {val:.4f}")

print_top_10(deg, "Degree Centrality")
print_top_10(bet, "Betweenness Centrality")
print_top_10(eig, "Eigenvector Centrality")

C:\Users\micro\AppData\Local\Temp\ipykernel_14564\2626367141.py:5: DtypeWarning: Columns (7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/enron.csv')


                      From                                                 To
0          msagel@home.com                                  jarnold@enron.com
1  slafontaine@globalp.com                              john.arnold@enron.com
2  iceoperations@intcx.com  icehelpdesk@intcx.com, internalmarketing@intcx...
3       klarnold@flash.net                              john.arnold@enron.com
4    soblander@carrfut.com                              soblander@carrfut.com

Computing centralities...

Top 10 Degree Centrality:
jeff.dasovich@enron.com: 0.0301
tana.jones@enron.com: 0.0230
sara.shackleton@enron.com: 0.0225
 : 0.0217
chris.germany@enron.com: 0.0198
klay@enron.com: 0.0180
kay.mann@enron.com: 0.0171
vince.kaminski@enron.com: 0.0162
gerald.nemec@enron.com: 0.0146
louise.kitchen@enron.com: 0.0129

Top 10 Betweenness Centrality:
john.lavorato@enron.com: 0.0447
jeff.dasovich@enron.com: 0.0387
louise.kitchen@enron.com: 0.0385
mark.taylor@enron.com: 0.0309
tana.jones@enron.com: 0.0287
steven